# Phase 7 — ML Problem Definition

## Objective

The goal of this phase is to formally define the machine learning problem based on the findings from the exploratory data analysis.

The selected research direction is:

**Refresh / Content Opportunity Scoring**

The research question is:

> Can historical search performance and content-level characteristics be used to rank content items by their priority for review or improvement?

The expected output is a ranked list of content items with:

- An opportunity or priority score
- The main reason for the score
- Supporting performance signals
- Supporting content-level characteristics

The model should identify content items that may deserve review or improvement based on observed historical data.

The analysis will not claim that improving a selected content item will directly cause higher Google rankings, impressions, or clicks.

## Task 7.1 — Define the ML Objective

### ML Objective

The objective of the machine learning component is to estimate the relative opportunity or priority of content items for review or improvement.

The model should use historical search-performance signals and content-level characteristics to identify content items that may have higher potential priority for review.

The output should support a ranked list rather than a simple binary decision.

### Unit of Prediction

The unit of prediction is:

**One content item for one client**

The analytical grain is:

`client_hash_id + content_hash_id`

Each row represents one content item associated with one client and contains aggregated historical search-performance metrics and content-level characteristics.

### Expected Model Output

For each content item, the final system should produce:

1. An opportunity or priority score.
2. A ranking relative to other content items.
3. Supporting signals explaining why the item received its score.

### Problem Formulation

The problem is treated as a **content opportunity ranking / scoring problem**.

The goal is not to predict a specific future Google ranking or traffic value directly.

Instead, the model should estimate which content items deserve higher priority for review based on the available historical evidence.

### Important Limitation

The available dataset does not provide a direct historical label indicating whether a content item was successfully improved and subsequently achieved better performance.

Therefore, the project should not invent a supervised target that claims to represent actual improvement success.

The target/scoring formulation will be defined in the next task based on the available historical performance data.

## Task 7.2 — Define the Target / Outcome

### Target Definition

The project does not contain a direct supervised label indicating whether a content item was successfully improved after optimization.

Therefore, a traditional binary target such as:

`improved = 1 / 0`

cannot be directly constructed from the available data without introducing unsupported assumptions.

Instead, the project will define a **continuous opportunity score** based on observable historical search-performance signals.

The score will represent the relative priority of a content item for review or improvement.

### Target Concept

The target should capture content items that show evidence of an opportunity based on a combination of:

- Search visibility
- Search clicks
- Search position
- Click-through behavior
- Reporting exposure
- Relevant content characteristics

The target is therefore interpreted as:

> **Relative content opportunity for review**

rather than:

> **Probability that an optimization will improve rankings or traffic**

### Target Type

The intended target is a:

**Continuous scoring outcome**

The output will be a numerical score that can be used to rank content items from higher to lower review priority.

### Why a Continuous Score?

A continuous score is appropriate because the project objective is ranking content items rather than assigning them to only two classes.

For example, the final system may produce:

| Content Item | Opportunity Score | Priority |
|---|---:|---|
| Content A | 0.91 | High |
| Content B | 0.74 | High |
| Content C | 0.52 | Medium |
| Content D | 0.21 | Low |

The exact scoring methodology and thresholds will be defined and validated in later phases.

### Target Limitation

The opportunity score should be interpreted as a prioritization signal derived from historical data.

It should not be interpreted as:

- A guaranteed prediction of future traffic
- A guaranteed improvement in Google rankings
- A causal estimate of the effect of optimization
- A probability of successful content improvement

The score indicates which content items may deserve attention based on the evidence available in the dataset.

## Task 7.3 — Define the Problem Type

### Problem Type

The primary machine learning problem is defined as a:

**Ranking / Scoring Problem**

The objective is to assign each content item a numerical opportunity score and use that score to rank content items according to their relative priority for review.

### Why Ranking / Scoring?

The research question asks which content items should receive higher priority for review or improvement.

This is fundamentally different from a standard classification problem because the desired output is not simply:

`High / Low`

Instead, the system should preserve relative differences between content items.

A continuous score allows content items to be ordered and prioritized.

### Relationship to Supervised Learning

A direct supervised learning target is not currently available because the dataset does not contain a historical outcome showing whether an optimization intervention successfully improved a content item's performance.

Therefore, the project will initially focus on constructing a data-driven opportunity score from observable historical evidence.

Machine learning models can then be evaluated for their ability to reproduce or improve this ranking framework without introducing unsupported labels.

### Final Problem Statement

> Given historical search-performance data and content-level characteristics for each content item, estimate a relative opportunity score that can be used to rank content items by their priority for review or improvement.

## Task 7.4 — Define Input Features

### Candidate Input Features

The candidate features are divided into three main groups:

#### 1. Historical Search Performance Features

These features describe the observed search performance of each content item during the historical reporting period.

| Feature | Description | Role |
|---|---|---|
| `total_gsc_impressions` | Total search impressions | Performance signal |
| `total_gsc_clicks` | Total search clicks | Performance signal |
| `mean_gsc_avg_position` | Mean observed search position | Performance signal |
| `reporting_days` | Number of days with available performance records | Exposure / reliability signal |

These features provide direct evidence about historical search visibility and traffic.

#### 2. Content-Level Features

These features describe the characteristics of the content item.

| Feature | Description | Role |
|---|---|---|
| `content_type` | Type of content | Content characteristic |
| `search_volume` | Associated search volume | Search-demand signal |
| `competition` | Numerical competition measure | Search environment |
| `competition_level` | Categorical competition level | Search environment |
| `cpc` | Cost-per-click indicator | Search-demand signal |
| `main_intent` | Search intent category | Content/search characteristic |
| `backlinks` | Number of backlinks | Content authority signal |
| `category_count` | Number of associated categories | Content characteristic |
| `word_count` | Content word count | Content characteristic |
| `char_count` | Content character count | Content characteristic |

These features may help explain differences in observed content performance.

#### 3. Content Status / Timing Features

The following variables may provide additional information about the content lifecycle:

- `last_optimized_date`
- `optimization_eligible_date`
- `is_published`
- `is_deleted`

These variables require additional investigation before being included because their meaning may introduce temporal or business-process effects.

### Features Requiring Special Treatment

Some variables contain substantial missing values or highly skewed distributions.

Examples include:

- `backlinks`
- `word_count`
- `char_count`
- `search_volume`
- `main_intent`
- `competition_level`
- `mean_gsc_avg_position`

These features should not automatically be removed.

Their missingness, distributions, and appropriate transformations will be addressed during the feature-engineering phase.

### Identifier Columns

The following columns identify the content or client but should not be treated as ordinary predictive numerical features:

- `client_hash_id`
- `content_hash_id`

They will be retained for grouping, tracking, ranking, and final output, but should not be directly used as model features.

### Initial Feature Set

The initial candidate feature set therefore consists of:

**Performance:**
- `total_gsc_impressions`
- `total_gsc_clicks`
- `mean_gsc_avg_position`
- `reporting_days`

**Content:**
- `content_type`
- `search_volume`
- `competition`
- `competition_level`
- `cpc`
- `main_intent`
- `backlinks`
- `category_count`
- `word_count`
- `char_count`

Additional status and timing variables will only be included if their temporal role is confirmed during feature engineering.

## Task 7.5 — Data Leakage Risks

### Potential Leakage Sources

Data leakage must be carefully controlled because the opportunity score is derived from historical performance.

#### 1. Using Future Information

Features must only use information that would have been available at the time the content item is evaluated.

Information from a future period must not be used to construct features for an earlier evaluation point.

#### 2. Target-Derived Features

If the opportunity score is constructed from performance variables such as:

- impressions
- clicks
- CTR
- average position

the same information must not be reused improperly as an independent feature when evaluating a model against that score.

The relationship between the target construction and model features must therefore be explicitly documented.

#### 3. Aggregation Leakage

Daily performance records must be aggregated carefully.

Aggregating information from the complete observation period and then treating it as if it were available at the beginning of that period would introduce temporal leakage.

#### 4. Identifier Leakage

`client_hash_id` and `content_hash_id` should not be treated as predictive numerical variables.

They identify entities rather than represent meaningful content characteristics.

#### 5. Post-Outcome Information

Variables representing events that occurred after the evaluation point should not be used as predictive features.

Examples include information that becomes available only after a content item has been reviewed, optimized, or otherwise changed.

### Leakage Prevention Principle

The modeling dataset should follow this principle:

> Only information available before or at the defined evaluation point may be used to construct features for that evaluation.

Temporal ordering and feature availability will therefore be explicitly considered during feature engineering and model validation.

## Task 7.6 — Evaluation Strategy

### Evaluation Objective

The evaluation strategy should measure whether the resulting scoring system can produce a meaningful ranking of content items according to their relative opportunity.

Since the project is formulated as a ranking/scoring problem, classification accuracy is not an appropriate primary evaluation metric.

### Primary Evaluation Perspective

The evaluation will focus on whether high-priority content items are concentrated near the top of the generated ranking.

The ranking should therefore be evaluated using metrics that measure the quality of the top-ranked items.

### Candidate Ranking Metrics

The following metrics will be considered:

#### 1. Precision@K

Measures the proportion of relevant content items among the top K ranked items.

This is useful because the practical use case is expected to focus on a limited number of content items for review.

#### 2. Recall@K

Measures how much of the relevant opportunity set is captured within the top K ranked items.

This provides a complementary view to Precision@K.

#### 3. NDCG@K

Normalized Discounted Cumulative Gain evaluates ranking quality while giving greater importance to highly relevant items appearing near the top of the ranking.

NDCG@K is particularly relevant when content items can have different degrees of opportunity rather than only a binary relevant/not-relevant label.

### Secondary Evaluation

Where appropriate, numerical prediction error may also be examined if a continuous target is used.

Possible metrics include:

- MAE
- RMSE

However, these metrics should not replace ranking-based evaluation because the main objective is to prioritize content items.

### Top-K Evaluation

The final system should be evaluated at practical ranking levels such as:

- Top 10
- Top 25
- Top 50
- Top 100

The exact K values may be adjusted depending on the final dataset size and business use case.

### Baseline Comparison

The proposed scoring approach should be compared with simple baseline ranking strategies.

Possible baselines include:

- Ranking by total impressions
- Ranking by total clicks
- Ranking by average position
- A simple rule-based opportunity score

The purpose of the baseline is to determine whether the proposed approach provides additional ranking value beyond simple performance-based ordering.

### Evaluation Principle

The evaluation should answer the following question:

> Does the proposed scoring approach identify a more useful set of high-priority content items than simple baseline ranking strategies?

The evaluation results will be interpreted as evidence about ranking quality, not as evidence that optimization will causally improve search performance.

## Task 7.7 — Define the Evaluation Reference

### Evaluation Reference

The dataset does not contain a historical ground-truth label indicating whether a content item was successfully improved after optimization.

Therefore, the project will use a **reference opportunity score** derived from observable historical signals.

The reference score will be used as an evaluation benchmark rather than as evidence of future optimization success.

### Reference Score Concept

The reference score should represent the relative priority of a content item based on observable evidence such as:

- Search visibility
- Search traffic
- Search position
- Click-through behavior
- Sufficient reporting exposure

The score should favor content items where there is meaningful evidence of search visibility or demand while also indicating potential room for review.

### Important Constraint

The reference score must not simply reproduce one individual feature such as:

- total impressions
- total clicks
- average position
- CTR

Using one performance metric directly as the complete target would make the resulting evaluation largely circular.

Instead, the reference score should combine multiple independent signals into a transparent scoring framework.

### Exposure Requirement

Content items with extremely limited reporting exposure should not automatically receive a high opportunity score because their observed performance may be less reliable.

`reporting_days` will therefore be considered when determining the reliability of the observed signals.

### Zero-Visibility Content

Content items with zero impressions require special treatment.

A content item with no observed impressions provides limited evidence about its search performance during the observation period.

Therefore, zero-impression content should not automatically be interpreted as a high-opportunity item.

Such content may represent:

- Low search visibility
- Insufficient search exposure
- Missing or limited search demand
- Other factors that cannot be distinguished from the available data

### Interpretation

The reference score should therefore be interpreted as:

> A transparent, data-driven measure of relative review priority based on observed historical search signals.

It should not be interpreted as:

> The probability that optimization will improve rankings, impressions, or clicks.

### Future Validation

If future data becomes available after content changes are made, the reference framework can be evaluated against actual post-intervention outcomes.

This would provide a stronger supervised learning target for future versions of the project.

## Task 7.8 — Final ML Problem Definition

### Final ML Problem

The project is formulated as a **content opportunity ranking and scoring problem**.

Given a set of content items associated with individual clients, the system will use historical search-performance signals and content-level characteristics to estimate a relative opportunity score.

The resulting score will be used to rank content items according to their priority for review or potential improvement.

### Input

The model will use candidate features representing:

- Historical search performance
- Search visibility
- Search traffic
- Search position
- Reporting exposure
- Content characteristics
- Search intent
- Competition
- Content size
- Backlink information
- Other validated content-level signals

### Unit of Analysis

The unit of analysis is:

`client_hash_id + content_hash_id`

Each row represents one content item for one client.

### Output

The system will produce:

1. A continuous opportunity score.
2. A ranking of content items.
3. Supporting signals or reason codes explaining the ranking.

### Problem Type

**Ranking / Scoring**

The primary objective is to prioritize content items rather than classify them into fixed categories.

### Evaluation

The ranking will be evaluated using top-K ranking metrics such as:

- Precision@K
- Recall@K
- NDCG@K

Simple ranking baselines will also be used for comparison.

### Target Limitation

There is no direct historical label indicating successful content improvement.

Therefore, the project will use a transparent reference opportunity score derived from observable historical signals for the initial evaluation framework.

This reference score is not a causal or predictive measure of future optimization success.

### Leakage Prevention

Only information that would be available at the defined evaluation point should be used to construct model features.

Future performance information or post-intervention information must not be used as predictive input.

### Final Research-Oriented Problem Statement

> Given historical search-performance data and content-level characteristics for each content item, can a data-driven scoring approach rank content items according to their relative priority for review or improvement?

The resulting ranking should provide a practical prioritization framework while avoiding unsupported claims about causal improvements in search performance.